# 02 — Re-check and supplementary checks (no LLM calls)

Reproduces the checks added in the revision. Run the cells in order; every output CSV is written to `notebooks/outputs/`
(the copies used in the paper are already in `data/combined/` and `data/supplementary/`).

| Step | Paper section | Output |
|---|---|---|
| 1. Re-check the 30 failed Docker Hub lookups | III-D | `recheck_30_rows.csv`, `combined_full_results_rechecked.csv` |
| 2. Validate the 56 references to other registries | IV-A | `registry_check_56.csv` |
| 3. Namespace existence and repository maintenance | IV-C | `risk_namespaces.csv`, `risk_hub_metadata.csv`, `risk_per_response.csv` |
| 4. Trivy CVE scan (v0.76.2, slow) | IV-C | `cve_scan_results.csv`, `cve_per_response.csv` |
| 5. Overlapping issues and hadolint baseline (Table III) | IV-C | `flags.csv` |
| 6. Outdated Tags vs. knowledge cutoff | IV-B, IV-F | printed |

Open this notebook from a clone of the repository (in Colab: `!git clone <repo-url>` then `%cd <repo>/notebooks`).

In [ ]:
import os
os.makedirs("outputs", exist_ok=True)
os.chdir("outputs")          # inputs are read from ../../data, outputs are written here
print(os.getcwd())

## 1. Re-check the 30 rows whose Docker Hub lookup failed

In [ ]:
# ==========================================================================
# RE-CHECK THE 30 ROWS WHOSE DOCKER HUB LOOKUP FAILED
# - No LLM calls. Only Docker Hub (hub API + registry) is queried.
# - Same category names and priority rule as the main pipeline.
# - Run in a fresh Colab cell. Takes about 2-5 minutes.
# ==========================================================================
import os, re, time, datetime as dt
import requests
import pandas as pd

FILE  = "../../data/combined/combined_final_report_before_recheck.xlsx"   # results before the re-check
SHEET = "Combined Full Results"

if not os.path.exists(FILE):
    from google.colab import files
    print("Please upload 'combined_final_report (1).xlsx' ...")
    uploaded = files.upload()
    FILE = next(iter(uploaded))

df = pd.read_excel(FILE, sheet_name=SHEET)
failed = (df["Automated Category"] == "NEEDS MANUAL CHECK") & \
         df["Automated Note"].astype(str).str.startswith("Docker Hub lookup failed")
print(f"Rows to re-check: {failed.sum()} (expected 30)")

CHECK_TIME = dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%d %H:%M UTC")

# Products the main pipeline maps to endoflife.date (copied from the notebook)
EOL_PRODUCT_MAP = {
    "node", "python", "golang", "go", "php", "ruby", "rust", "postgres", "mysql", "redis",
    "mongo", "rabbitmq", "nginx", "eclipse-temurin", "mariadb", "httpd", "wordpress",
    "drupal", "elixir", "erlang", "haskell", "grafana/grafana", "prom/prometheus",
    "jenkins/jenkins", "jenkins", "cassandra", "solr", "haproxy", "r-base",
}

session = requests.Session()
session.headers["User-Agent"] = "docker-hallucination-study-recheck"


def http(url, method="GET", headers=None, tries=6):
    """Request with polite pacing and long back-off on rate limits (429) and server errors."""
    wait = 10
    for _ in range(tries):
        try:
            r = session.request(method, url, headers=headers, timeout=30)
        except requests.RequestException:
            time.sleep(wait); wait = min(wait * 2, 120); continue
        if r.status_code == 429 or r.status_code >= 500:
            retry_after = r.headers.get("Retry-After")
            reset = r.headers.get("X-RateLimit-Reset")
            if retry_after and retry_after.isdigit():
                pause = int(retry_after)
            elif reset and reset.isdigit():
                pause = max(1, int(reset) - int(time.time()))
            else:
                pause = wait
            pause = min(pause, 180)
            print(f"   rate-limited/server error ({r.status_code}), waiting {pause}s ...")
            time.sleep(pause); wait = min(wait * 2, 120); continue
        time.sleep(1.5)  # pacing between successful calls
        return r.status_code, r
    return None, None


def split_ref(image):
    """'nginx' -> ('library','nginx',True); 'nvidia/cuda' -> ('nvidia','cuda',False)"""
    parts = image.strip().split("/")
    if len(parts) == 1:
        return "library", parts[0], True
    return parts[0], "/".join(parts[1:]), False


_cache = {}

def check(image, tag):
    key = (image, tag)
    if key in _cache:
        return _cache[key]
    ns, repo, official = split_ref(image)

    hub_img, _ = http(f"https://hub.docker.com/v2/repositories/{ns}/{repo}/")
    hub_tag, _ = http(f"https://hub.docker.com/v2/repositories/{ns}/{repo}/tags/{tag}/")

    # Second opinion from the registry itself (HEAD requests do not count toward pull limits)
    reg = None
    st, tok = http(f"https://auth.docker.io/token?service=registry.docker.io"
                   f"&scope=repository:{ns}/{repo}:pull")
    if st == 200:
        accept = ", ".join([
            "application/vnd.oci.image.index.v1+json",
            "application/vnd.docker.distribution.manifest.list.v2+json",
            "application/vnd.docker.distribution.manifest.v2+json",
            "application/vnd.oci.image.manifest.v1+json",
        ])
        reg, _ = http(f"https://registry-1.docker.io/v2/{ns}/{repo}/manifests/{tag}",
                      method="HEAD",
                      headers={"Authorization": f"Bearer {tok.json().get('token')}",
                               "Accept": accept})

    # Decide existence. 200 from either source = exists; 404 from the hub API = missing
    # (the registry answers 401 for repositories that do not exist).
    if hub_img == 200 or reg == 200:
        image_exists = True
    elif hub_img == 404:
        image_exists = False
    else:
        image_exists = None

    if image_exists is False:
        tag_exists = False
    elif hub_tag == 200 or reg == 200:
        tag_exists = True
    elif hub_tag == 404 and reg != 200:
        tag_exists = False
    else:
        tag_exists = None

    # Same priority rule as the main pipeline
    if image_exists is None or tag_exists is None:
        cat, note = "NEEDS MANUAL CHECK", "Docker Hub lookup failed again on re-check"
    elif not image_exists:
        cat, note = "Non-Existent Image", "image not found on Docker Hub"
    elif not tag_exists:
        cat, note = "Nonexistent Tag", "tag not found on Docker Hub"
    elif not official:
        cat, note = "Non-Official Image", f"namespace '{ns}' - not Docker Official Images"
    elif tag.strip().lower() == "latest":
        cat, note = "Risky Latest Tag", "tag is literally 'latest'"
    elif repo in EOL_PRODUCT_MAP:
        cat, note = "NEEDS MANUAL CHECK", "exists; run the notebook's EOL check for this row"
    else:
        cat, note = ("NEEDS MANUAL CHECK",
                     f"exists; no endoflife.date mapping configured for image '{repo}'")

    out = dict(hub_image_status=hub_img, hub_tag_status=hub_tag, registry_status=reg,
               image_exists=image_exists, tag_exists=tag_exists,
               new_category=cat, new_note=f"{note} (re-checked {CHECK_TIME})")
    _cache[key] = out
    return out


rows = []
for idx, r in df[failed].iterrows():
    image, tag = str(r["Image Name"]).strip(), str(r["Tag"]).strip()
    print(f"Checking {image}:{tag} ...")
    res = check(image, tag)
    rows.append({"row": idx, "LLM": r["LLM"], "Scenario": r["Scenario"],
                 "Image Name": image, "Tag": tag,
                 "old_category": r["Automated Category"], **res})

recheck = pd.DataFrame(rows)
pd.set_option("display.width", 250, "display.max_colwidth", 60)
print("\n=== Re-check results (unique image:tag) ===")
print(recheck.drop_duplicates(["Image Name", "Tag"])[
    ["Image Name", "Tag", "hub_image_status", "hub_tag_status", "registry_status", "new_category"]
].to_string(index=False))
print("\nNew categories for the 30 rows:")
print(recheck["new_category"].value_counts().to_string())

# Apply to the full table and recompute the paper's numbers
updated = df.copy()
for _, r in recheck.iterrows():
    updated.at[r["row"], "Automated Category"] = r["new_category"]
    updated.at[r["row"], "Automated Note"] = r["new_note"]

in_scope = updated[updated["Automated Category"] != "OUT OF SCOPE"]
summary = in_scope["Automated Category"].value_counts().rename("Count").to_frame()
summary["%"] = (100 * summary["Count"] / len(in_scope)).round(1)
print(f"\n=== Updated overall distribution (in-scope n = {len(in_scope)}) ===")
print(summary.to_string())
print("\n=== Updated per-batch % ===")
print((100 * pd.crosstab(in_scope["Batch"], in_scope["Automated Category"], normalize="index")).round(1).to_string())
print("\n=== Updated per-LLM counts ===")
print(pd.crosstab(in_scope["LLM"], in_scope["Automated Category"]).to_string())

recheck.to_csv("recheck_30_rows.csv", index=False)
updated.to_csv("combined_full_results_rechecked.csv", index=False)
print(f"\nSaved recheck_30_rows.csv and combined_full_results_rechecked.csv (checked {CHECK_TIME})")

try:
    from google.colab import files
    files.download("recheck_30_rows.csv")
    files.download("combined_full_results_rechecked.csv")
except Exception:
    pass


## 2. References to registries other than Docker Hub

In [ ]:
# ==========================================================================
# CELL A - Issue 3: validate the 56 references to registries other than Docker Hub
# (gcr.io, quay.io, mcr.microsoft.com, docker.elastic.co). No LLM calls.
# ==========================================================================
import os, re, time, datetime as dt
import requests, pandas as pd

FILE = "../../data/combined/combined_full_results_rechecked.csv"
if not os.path.exists(FILE):
    from google.colab import files
    print(f"Please upload {FILE} ...")
    FILE = next(iter(files.upload()))

df = pd.read_csv(FILE)
oos = df[df["Automated Category"] == "OUT OF SCOPE"].copy()
print(f"Out-of-scope rows: {len(oos)} (expected 56)")
CHECK_DATE = dt.date.today()
CHECK_TIME = dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%d %H:%M UTC")

S = requests.Session()
S.headers["User-Agent"] = "docker-hallucination-study-registry-check"
ACCEPT = ", ".join([
    "application/vnd.oci.image.index.v1+json",
    "application/vnd.docker.distribution.manifest.list.v2+json",
    "application/vnd.docker.distribution.manifest.v2+json",
    "application/vnd.oci.image.manifest.v1+json",
])


def http(url, method="GET", headers=None, params=None, tries=6):
    wait = 5
    for _ in range(tries):
        try:
            r = S.request(method, url, headers=headers, params=params, timeout=30)
        except requests.RequestException:
            time.sleep(wait); wait = min(wait * 2, 120); continue
        if r.status_code == 429 or r.status_code >= 500:
            ra = r.headers.get("Retry-After", "")
            pause = int(ra) if ra.isdigit() else wait
            print(f"   {r.status_code} from {url.split('/')[2]}, waiting {min(pause,180)}s")
            time.sleep(min(pause, 180)); wait = min(wait * 2, 120); continue
        time.sleep(1.0)
        return r
    return None


def registry_get(host, repo, path, method="GET"):
    """OCI distribution request with the anonymous bearer-token flow."""
    url = f"https://{host}/v2/{repo}/{path}"
    h = {"Accept": ACCEPT}
    r = http(url, method, headers=h)
    if r is not None and r.status_code == 405 and method == "HEAD":
        r = http(url, "GET", headers=h)
    if r is not None and r.status_code == 401 and "bearer" in r.headers.get("WWW-Authenticate", "").lower():
        auth = dict(re.findall(r'(\w+)="([^"]*)"', r.headers["WWW-Authenticate"]))
        params = {"scope": auth.get("scope", f"repository:{repo}:pull")}
        if "service" in auth:
            params["service"] = auth["service"]
        t = http(auth["realm"], params=params)
        tok = None
        if t is not None and t.status_code == 200:
            js = t.json()
            tok = js.get("token") or js.get("access_token")
        if tok:
            h["Authorization"] = f"Bearer {tok}"
            r = http(url, method, headers=h)
            if r is not None and r.status_code == 405 and method == "HEAD":
                r = http(url, "GET", headers=h)
    return r.status_code if r is not None else None


# endoflife.date products for these images (a 404 simply means "no lifecycle data")
EOL_PRODUCT = {
    "dotnet/aspnet": "dotnet", "elasticsearch/elasticsearch": "elasticsearch",
    "istio-release/pilot": "istio", "istio-release/mixer": "istio",
    "coreos/etcd": "etcd", "google_containers/etcd-amd64": "etcd",
}
_eol = {}

def eol_status(repo, tag):
    prod = EOL_PRODUCT.get(repo)
    if not prod:
        return "no EOL data", ""
    if prod not in _eol:
        r = http(f"https://endoflife.date/api/{prod}.json")
        _eol[prod] = r.json() if r is not None and r.status_code == 200 else None
    cycles = _eol[prod]
    m = re.match(r"^v?(\d+(?:\.\d+)*)", tag)
    if not cycles or not m:
        return "no EOL data", ""
    parts = m.group(1).split(".")
    for k in range(len(parts), 0, -1):          # most specific cycle first: 8.19.4 -> 8.19 -> 8
        cyc = ".".join(parts[:k])
        hit = [c for c in cycles if str(c.get("cycle")) == cyc]
        if hit:
            eol = hit[0].get("eol")
            if eol is True:
                return "EOL", f"cycle {cyc} marked eol=true"
            if isinstance(eol, str):
                passed = dt.date.fromisoformat(eol) < CHECK_DATE
                return ("EOL" if passed else "SUPPORTED"), f"cycle {cyc} eol {eol}"
            return "SUPPORTED", f"cycle {cyc} has no EOL date"
    return "no EOL data", f"version {m.group(1)} not in endoflife.date cycles"


_cache = {}

def check(image, tag):
    if (image, tag) in _cache:
        return _cache[(image, tag)]
    host, repo = image.split("/", 1)
    tag_st = registry_get(host, repo, f"manifests/{tag}", "HEAD")
    repo_st = registry_get(host, repo, "tags/list")
    eol, eol_note = ("", "")
    if tag_st == 200:
        if tag.lower() == "latest":
            cat = "Risky Latest Tag"
        else:
            eol, eol_note = eol_status(repo, tag)
            cat = {"EOL": "Outdated Tag", "SUPPORTED": "Valid (current)"}.get(eol, "Exists (no EOL data)")
    elif tag_st == 404 and repo_st == 200:
        cat = "Non-Existent Tag"
    elif tag_st == 404 and repo_st == 404:
        cat = "Non-Existent Image"
    elif tag_st in (401, 403) or repo_st in (401, 403):
        cat = "Non-existent or private (access denied)"
    elif tag_st == 404:
        cat = "Non-Existent Tag or Image"
    else:
        cat = "Lookup failed"
    out = dict(host=host, tag_status=tag_st, repo_status=repo_st, category=cat,
               eol_status=eol, eol_note=eol_note, checked=CHECK_TIME)
    _cache[(image, tag)] = out
    return out


rows = []
for idx, r in oos.iterrows():
    image, tag = str(r["Image Name"]).strip(), str(r["Tag"]).strip()
    print(f"Checking {image}:{tag}")
    rows.append({"row": idx, "LLM": r["LLM"], "Scenario": r["Scenario"], "Batch": r["Batch"],
                 "Image Name": image, "Tag": tag, **check(image, tag)})

res = pd.DataFrame(rows)
pd.set_option("display.width", 250, "display.max_colwidth", 60)
print("\n=== Unique image:tag results ===")
print(res.drop_duplicates(["Image Name", "Tag"])[
    ["Image Name", "Tag", "tag_status", "repo_status", "category", "eol_note"]].to_string(index=False))
print("\n=== Categories over the 56 responses ===")
print(res["category"].value_counts().to_string())
print("\n=== By registry ===")
print(pd.crosstab(res["host"], res["category"]).to_string())
res.to_csv("registry_check_56.csv", index=False)
print(f"\nSaved registry_check_56.csv (checked {CHECK_TIME})")
try:
    from google.colab import files; files.download("registry_check_56.csv")
except Exception:
    pass


## 3. Namespace existence and repository maintenance

In [ ]:
# ==========================================================================
# CELL B - Issue 6 (part 1): registrable namespaces and unmaintained images
# Docker Hub metadata only. No image pulls, no LLM calls. About 3-5 minutes.
# ==========================================================================
import os, time, datetime as dt
import requests, pandas as pd

FILE = "../../data/combined/combined_full_results_rechecked.csv"
if not os.path.exists(FILE):
    from google.colab import files
    print(f"Please upload {FILE} ...")
    FILE = next(iter(files.upload()))

df = pd.read_csv(FILE)
ins = df[df["Automated Category"] != "OUT OF SCOPE"].copy()
NOW = dt.datetime.now(dt.timezone.utc)
CHECK_TIME = NOW.strftime("%Y-%m-%d %H:%M UTC")
STALE_DAYS = 730   # "unmaintained" = no push for more than 2 years

S = requests.Session()
S.headers["User-Agent"] = "docker-hallucination-study-risk-check"


def http(url, tries=6):
    wait = 10
    for _ in range(tries):
        try:
            r = S.get(url, timeout=30)
        except requests.RequestException:
            time.sleep(wait); wait = min(wait * 2, 120); continue
        if r.status_code == 429 or r.status_code >= 500:
            ra = r.headers.get("Retry-After", ""); reset = r.headers.get("X-RateLimit-Reset", "")
            pause = int(ra) if ra.isdigit() else (max(1, int(reset) - int(time.time())) if reset.isdigit() else wait)
            print(f"   rate-limited ({r.status_code}), waiting {min(pause,180)}s")
            time.sleep(min(pause, 180)); wait = min(wait * 2, 120); continue
        time.sleep(1.2)
        return r
    return None


def split_ref(image):
    p = image.strip().split("/")
    return ("library", p[0]) if len(p) == 1 else (p[0], "/".join(p[1:]))


def age_days(ts):
    if not ts:
        return None
    return (NOW - pd.to_datetime(ts, utc=True)).days


# ---- 1. Registrable namespaces: does every recommended namespace exist? ----
namespaces = sorted({split_ref(i)[0] for i in ins["Image Name"]} - {"library"})
ns_rows = []
for ns in namespaces:
    r1 = http(f"https://hub.docker.com/v2/repositories/{ns}/?page_size=1")
    r2 = http(f"https://hub.docker.com/v2/users/{ns}/")
    st1 = r1.status_code if r1 is not None else None
    st2 = r2.status_code if r2 is not None else None
    exists = (st1 == 200) or (st2 == 200)
    unknown = not exists and (st1 not in (404,) or st2 not in (404,))
    ns_rows.append(dict(namespace=ns, repos_status=st1, users_status=st2,
                        exists=exists, registrable=(not exists and not unknown), unknown=unknown))
    print(f"namespace {ns}: repos={st1} users={st2} -> {'exists' if exists else ('UNKNOWN' if unknown else 'REGISTRABLE')}")
nsdf = pd.DataFrame(ns_rows)
reg_ns = set(nsdf.loc[nsdf["registrable"], "namespace"])

# ---- 2. Maintenance: repository last update and tag push date ----
meta = {}
for image, tag in ins[["Image Name", "Tag"]].drop_duplicates().itertuples(index=False):
    ns, repo = split_ref(image)
    rr = http(f"https://hub.docker.com/v2/repositories/{ns}/{repo}/")
    rt = http(f"https://hub.docker.com/v2/repositories/{ns}/{repo}/tags/{tag}/")
    rj = rr.json() if rr is not None and rr.status_code == 200 else {}
    tj = rt.json() if rt is not None and rt.status_code == 200 else {}
    tag_pushed = tj.get("tag_last_pushed") or tj.get("last_updated")
    meta[(image, tag)] = dict(
        repo_exists=(rr is not None and rr.status_code == 200), tag_exists=(rt is not None and rt.status_code == 200),
        repo_last_updated=rj.get("last_updated"), repo_age_days=age_days(rj.get("last_updated")),
        tag_last_pushed=tag_pushed, tag_age_days=age_days(tag_pushed),
        compressed_size_bytes=tj.get("full_size"), pull_count=rj.get("pull_count"))
    print(f"{image}:{tag} -> repo age {meta[(image, tag)]['repo_age_days']} d, tag age {meta[(image, tag)]['tag_age_days']} d")

md = pd.DataFrame([{"Image Name": k[0], "Tag": k[1], **v} for k, v in meta.items()])
md["repo_unmaintained"] = md["repo_age_days"] > STALE_DAYS
md["tag_older_than_2y"] = md["tag_age_days"] > STALE_DAYS

resp = ins.merge(md, on=["Image Name", "Tag"], how="left")
resp["namespace"] = resp["Image Name"].map(lambda i: split_ref(i)[0])
resp["registrable_namespace"] = resp["namespace"].isin(reg_ns)

print(f"\n=== Registrable namespaces: {len(reg_ns)} of {len(namespaces)} ===", sorted(reg_ns))
print(f"Responses pointing to a registrable namespace: {int(resp['registrable_namespace'].sum())} of {len(resp)}")
summary = resp.groupby("Automated Category").agg(
    responses=("Image Name", "size"),
    pct_repo_unmaintained=("repo_unmaintained", lambda s: round(100 * s.mean(), 1)),
    pct_tag_older_2y=("tag_older_than_2y", lambda s: round(100 * s.mean(), 1)),
    median_repo_age_days=("repo_age_days", "median"))
print("\n=== Maintenance by primary category (repository not updated / tag not pushed for > 2 years) ===")
print(summary.to_string())
print(f"\nAll responses: {100*resp['repo_unmaintained'].mean():.1f}% unmaintained repositories, "
      f"{100*resp['tag_older_than_2y'].mean():.1f}% tags older than 2 years")

nsdf.to_csv("risk_namespaces.csv", index=False)
md.to_csv("risk_hub_metadata.csv", index=False)       # also used by Cell C
resp.to_csv("risk_per_response.csv", index=False)
print(f"\nSaved risk_namespaces.csv, risk_hub_metadata.csv, risk_per_response.csv (checked {CHECK_TIME})")
try:
    from google.colab import files
    for f in ["risk_namespaces.csv", "risk_hub_metadata.csv", "risk_per_response.csv"]:
        files.download(f)
except Exception:
    pass


## 4. Trivy CVE scan (needs the outputs of step 3; takes 1–4 hours)

In [ ]:
# ==========================================================================
# CELL C - Issue 6 (part 2): known vulnerabilities (CVEs) with Trivy
# Pulls and scans each unique, existing Docker Hub image:tag. SLOW: 1-4 hours.
# Run Cell B first (it creates risk_hub_metadata.csv). Re-running this cell
# resumes where it stopped, as long as the Colab runtime was not reset.
# ==========================================================================
import os, json, shutil, subprocess, getpass, datetime as dt
import pandas as pd

MAX_GB = 3.0                       # skip images whose compressed size is above this
CACHE = "/content/trivy-cache"
OUT = "cve_scan_results.csv"

# 1. Install Trivy (official install script)
if shutil.which("trivy") is None:
    subprocess.run("curl -sfL https://raw.githubusercontent.com/aquasecurity/trivy/main/contrib/install.sh"
                   " | sh -s -- -b /usr/local/bin v0.76.2", shell=True, check=True)
print(subprocess.run(["trivy", "--version"], capture_output=True, text=True).stdout.splitlines()[0])

# 2. Optional: log in to Docker Hub (a free account raises the anonymous pull limit)
user = input("Docker Hub username (press Enter to skip): ").strip()
if user:
    os.environ["TRIVY_USERNAME"] = user
    os.environ["TRIVY_PASSWORD"] = getpass.getpass("Docker Hub access token or password: ")

# 3. Images to scan
md = pd.read_csv("risk_hub_metadata.csv")
md = md[md["tag_exists"] == True].copy()
md["size_gb"] = md["compressed_size_bytes"].fillna(0) / 1e9
todo = md[md["size_gb"] <= MAX_GB]
skipped = md[md["size_gb"] > MAX_GB]
print(f"Images to scan: {len(todo)}; skipped as larger than {MAX_GB} GB: {len(skipped)}")
if len(skipped):
    print(skipped[["Image Name", "Tag", "size_gb"]].round(2).to_string(index=False))

done = pd.read_csv(OUT) if os.path.exists(OUT) else pd.DataFrame()
done_keys = set(zip(done.get("Image Name", []), done.get("Tag", [])))


def scan(ref):
    p = subprocess.run(["trivy", "image", "--quiet", "--scanners", "vuln", "--severity", "HIGH,CRITICAL",
                        "--format", "json", "--timeout", "20m", "--cache-dir", CACHE, ref],
                       capture_output=True, text=True)
    if p.returncode != 0:
        return None, p.stderr.strip().splitlines()[-1][:200] if p.stderr.strip() else "trivy error"
    data = json.loads(p.stdout or "{}")
    ids = {"HIGH": set(), "CRITICAL": set()}
    for res in data.get("Results", []) or []:
        for v in res.get("Vulnerabilities", []) or []:
            if v.get("Severity") in ids:
                ids[v["Severity"]].add(v.get("VulnerabilityID"))
    return {"high": len(ids["HIGH"]), "critical": len(ids["CRITICAL"])}, ""


for n, (image, tag) in enumerate(todo[["Image Name", "Tag"]].itertuples(index=False), 1):
    if (image, tag) in done_keys:
        continue
    ref = f"{image}:{tag}"
    print(f"[{n}/{len(todo)}] scanning {ref} ...", flush=True)
    counts, err = scan(ref)
    row = {"Image Name": image, "Tag": tag, "high": None, "critical": None, "error": err,
           "scanned": dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%d %H:%M UTC")}
    if counts:
        row.update(counts)
    pd.DataFrame([row]).to_csv(OUT, mode="a", header=not os.path.exists(OUT), index=False)
    shutil.rmtree(os.path.join(CACHE, "fanal"), ignore_errors=True)   # free disk; keep the vuln DB

# 4. Summary per primary category (each response counted once)
cve = pd.read_csv(OUT)
resp = pd.read_csv("risk_per_response.csv").merge(cve, on=["Image Name", "Tag"], how="left")
scanned = resp[resp["critical"].notna()]
summary = scanned.groupby("Automated Category").agg(
    responses_scanned=("Image Name", "size"),
    median_high=("high", "median"), median_critical=("critical", "median"),
    pct_with_critical=("critical", lambda s: round(100 * (s > 0).mean(), 1)))
print("\n=== High/Critical CVEs by primary category ===")
print(summary.to_string())
print(f"\nScanned responses: {len(scanned)} of {len(resp)}; scan errors: {int(cve['error'].fillna('').ne('').sum())}")
resp.to_csv("cve_per_response.csv", index=False)
try:
    from google.colab import files
    files.download(OUT); files.download("cve_per_response.csv")
except Exception:
    pass


## 5. Overlapping issues and hadolint baseline (Table III)

In [ ]:
# Install hadolint 2.15.1 (Linux x86_64)
import subprocess, shutil
if shutil.which("hadolint") is None:
    subprocess.run("wget -q -O /usr/local/bin/hadolint https://github.com/hadolint/hadolint/releases/download/v2.15.1/hadolint-Linux-x86_64 && chmod +x /usr/local/bin/hadolint", shell=True, check=True)
print(subprocess.run(["hadolint", "--version"], capture_output=True, text=True).stdout)

In [ ]:
import sys
sys.argv = ["overlap_hadolint", "../../data/combined/combined_full_results_rechecked.csv"]
import sys, json, subprocess, tempfile, os
import pandas as pd
c = pd.read_csv(sys.argv[1])
ins = c[c["Automated Category"] != "OUT OF SCOPE"].copy().reset_index(drop=True)
cat = ins["Automated Category"]
ins["nonexist"] = cat.isin(["Nonexistent Tag", "Non-Existent Image"])
ins["nonofficial"] = ins["Image Name"].astype(str).str.contains("/")
ins["latest"] = ins["Tag"].astype(str).str.strip().str.lower().eq("latest")
ins["outdated"] = cat.eq("Outdated Tag")
ins["unresolved"] = cat.eq("NEEDS MANUAL CHECK")

# ---- hadolint on each FROM line ----
codes = []
with tempfile.TemporaryDirectory() as d:
    for i, r in ins.iterrows():
        p = os.path.join(d, f"Dockerfile{i}")
        open(p, "w").write(f"FROM {r['Image Name']}:{r['Tag']}\n")
        out = subprocess.run(["hadolint", "--no-fail", "-f", "json", p], capture_output=True, text=True).stdout
        codes.append(sorted({x["code"] for x in json.loads(out or "[]")}))
ins["hadolint"] = codes
ins["hl_flag"] = ins["hadolint"].str.len() > 0
print("hadolint codes:", pd.Series([c for cs in codes for c in cs]).value_counts().to_dict())

prim = {"nonexist": cat.isin(["Nonexistent Tag","Non-Existent Image"]).sum(),
        "nonofficial": cat.eq("Non-Official Image").sum(),
        "latest": cat.eq("Risky Latest Tag").sum(),
        "outdated": cat.eq("Outdated Tag").sum()}
rows = []
for k, name in [("nonexist","Non-existent image/tag"),("nonofficial","Non-official namespace"),("latest","Unpinned latest"),("outdated","Outdated (EOL) version")]:
    m = ins[k]
    rows.append([name, prim[k], int(m.sum()), int((m & ins["hl_flag"]).sum())])
t = pd.DataFrame(rows, columns=["Issue","Primary label","Counted independently","Flagged by hadolint"])
print(t.to_string(index=False))

issues = ins[["nonexist","nonofficial","latest","outdated"]].sum(axis=1)
print("responses with >=2 issues:", int((issues>=2).sum()), "of", len(ins))
print("combos:", ins.loc[issues>=2, ["nonexist","nonofficial","latest","outdated"]].value_counts().to_dict())
nonvalid = ins[~cat.eq("Valid Official Image")]
res = nonvalid[~nonvalid["unresolved"]]
print("non-valid", len(nonvalid), "resolved non-valid", len(res), "hadolint flags", int(res["hl_flag"].sum()), "misses", int((~res["hl_flag"]).sum()))
miss = res[~res["hl_flag"]]
print("missed by category:", miss["Automated Category"].value_counts().to_dict())
print("valid flagged by hadolint (false alarms):", int(ins[cat.eq("Valid Official Image")]["hl_flag"].sum()))
print("unresolved flagged:", int(ins[ins['unresolved']]['hl_flag'].sum()), "of", int(ins['unresolved'].sum()))
fab = ins[ins["nonexist"] & ~ins["latest"]]
print("non-existent & not latest (fabrications):", fab[["LLM","Image Name","Tag"]].to_string(index=False))
ins.to_csv("flags.csv", index=False)


## 6. Outdated Tags vs. documented knowledge cutoffs (needs `flags.csv` from step 5)

In [ ]:
import pandas as pd
f = pd.read_csv("flags.csv"); o = f[f["Automated Category"] == "Outdated Tag"].copy()
o["eol"] = pd.to_datetime(o["Automated Note"].str.extract(r"eol date (\d{4}-\d{2}-\d{2})")[0])
# Documented knowledge cutoffs (provider docs). Month-only cutoffs -> end of that month.
CUTOFF = {"ChatGPT": "2026-02-16", "Claude": "2026-01-31", "Gemini": "2026-03-31", "GPT-OSS-120B": "2024-06-01"}
# No documented cutoff: public release date is an upper bound (DeepSeek-V4-Flash preview, 24 Apr 2026)
UPPER = {"DeepSeek": "2026-04-24"}
def place(r):
    if pd.isna(r.eol): return "undetermined (no EOL date)"
    if r.LLM in CUTOFF:
        return "EOL before cutoff" if r.eol <= pd.Timestamp(CUTOFF[r.LLM]) else "EOL after cutoff"
    if r.LLM in UPPER and r.eol > pd.Timestamp(UPPER[r.LLM]):
        return "EOL after cutoff"
    return "undetermined (no documented cutoff)"
o["placement"] = o.apply(place, axis=1)
print(o[["LLM","Image Name","Tag","eol","placement"]].sort_values(["placement","LLM"]).to_string(index=False))
print(o["placement"].value_counts().to_string())
print(pd.crosstab(o["LLM"], o["placement"]).to_string())
